# 01 - HR policy question: hybrid vs advanced RAG and the version trap

**Scenario.** Priya, a new employee at Aranya Technologies, asks: *"How many days of casual leave do I get per year,
and can I carry them forward?"* The HR store holds **two versions** of the Leave Policy: v2.0 (superseded) and v3.0
(current). A naive search can answer from the old version.

**You will see**
1. Hybrid RAG (BM25 + vectors + RRF) answering without a version filter.
2. Advanced RAG (rewrite -> current versions only -> LLM rerank -> ordered context).
3. An explicit question about the *old* version, answered from v2.0 on purpose.
4. The judge LLM scoring an answer.
5. The full multi-agent graph answering the same question.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

In [1]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

Coordinator: gpt-5.1 (high) | specialists: gpt-5.1 (low) | judge: gpt-5.5 (pass >= 80)


## 1. Hybrid RAG - no version awareness
Look at the `label` of each source: superseded chunks are marked `SUPERSEDED`.

In [2]:
from agentic_rag.rag import hybrid_rag, advanced_rag

QUESTION = "How many days of casual leave do I get per year, and can I carry them forward?"

hybrid = hybrid_rag(settings, "hr", QUESTION)
print("\n".join(hybrid.trace))
for s in hybrid.sources:
    print(f"[{s.source_id}] {s.label}")
print("\nANSWER:", hybrid.answer)

filter: none
hybrid (similarity + bm25, RRF): 6 chunks from HR-POL-001-V2.0, HR-POL-001-V3.0
context: 6 sources, 1275 tokens, 0 dropped
[S1] HR-POL-001-V2.0 p.2 | 5. Leave Entitlements | v2.0 | SUPERSEDED
[S2] HR-POL-001-V2.0 p.2 | 5. Leave Entitlements | v2.0 | SUPERSEDED
[S3] HR-POL-001-V2.0 p.3 | 10. Frequently Asked Questions | v2.0 | SUPERSEDED
[S4] HR-POL-001-V3.0 p.2 | 5. Leave Entitlements | v3.0
[S5] HR-POL-001-V3.0 p.2 | 5. Leave Entitlements | v3.0
[S6] HR-POL-001-V3.0 p.3 | 11. Frequently Asked Questions | v3.0

ANSWER: You get 8 days of Casual Leave (CL) per year under the current Leave Policy v3.0, and none of these days can be carried forward; any unused CL lapses on 31 December each year.

Details:
- Under the current Leave Policy v3.0, the annual entitlement for Casual Leave is 8 days, credited on 1 January each year, and it explicitly lapses on 31 December, meaning it cannot be carried forward or encashed. [S5]
- Under the older, superseded Leave Policy v2.0, the Casu

## 2. Advanced RAG - rewrite, current versions only, rerank

In [3]:
advanced = advanced_rag(settings, "hr", QUESTION)
print("\n".join(advanced.trace))
for s in advanced.sources:
    print(f"[{s.source_id}] {s.label}")
print("\nANSWER:", advanced.answer)

rewrite: 'casual leave entitlement per year and carry forward rules in Aranya Technologies latest leave policy'
version: current policies only (no old version requested)
hybrid candidates: 20 (filter: {'status': 'current'})
rerank: HR-POL-001-V3.0 p.2=10, HR-POL-001-V3.0 p.3=8, HR-POL-001-V3.0 p.2=3, HR-POL-001-V3.0 p.1=2, HR-POL-001-V3.0 p.2=2, HR-POL-001-V3.0 p.1=1
context: 6 sources, 1239 tokens, 0 dropped
[S1] HR-POL-001-V3.0 p.1 | Front matter | v3.0
[S2] HR-POL-001-V3.0 p.1 | 4. Leave Year and Holidays | v3.0
[S3] HR-POL-001-V3.0 p.2 | 5. Leave Entitlements | v3.0
[S4] HR-POL-001-V3.0 p.2 | 5. Leave Entitlements | v3.0
[S5] HR-POL-001-V3.0 p.2 | 6. Statutory and Special Leave | v3.0
[S6] HR-POL-001-V3.0 p.3 | 11. Frequently Asked Questions | v3.0

ANSWER: You get 8 days of Casual Leave (CL) per year, credited on 1 January each leave year, and any unused Casual Leave **cannot** be carried forward; it lapses on 31 December and is not encashed.[S2][S4][S6]


## 3. Asking about the old version on purpose
A version mention such as `v2.0` next to a policy title becomes a `doc_id` filter for that exact version.

In [4]:
old = advanced_rag(settings, "hr", "How many casual leave days per year did Leave Policy v2.0 give?")
print("\n".join(old.trace))
print("\nANSWER:", old.answer)

rewrite: 'casual leave entitlement per year in Leave Policy v2.0 at Aranya Technologies, number of casual leave days annually'
hybrid candidates: 14 (filter: {'doc_id': 'HR-POL-001-V2.0'})
rerank: HR-POL-001-V2.0 p.2=10, HR-POL-001-V2.0 p.3=3, HR-POL-001-V2.0 p.2=3, HR-POL-001-V2.0 p.1=2, HR-POL-001-V2.0 p.2=2, HR-POL-001-V2.0 p.1=1
version: added 2 chunks from current version HR-POL-001-V3.0 for comparison
context: 8 sources, 1679 tokens, 0 dropped

ANSWER: Leave Policy v2.0 provided 7 days of Casual Leave (CL) per year. [S4]

Note: Leave Policy v2.0 is now superseded by Leave Policy v3.0 and must not be used for current decisions. [S1][S7]


## 4. The judge
The judge (`JUDGE_OPENAI_MODEL`, its own API key) scores context relevance, groundedness, answer relevance and completeness; the weighted overall must reach `JUDGE_PASS_SCORE`.

In [5]:
from agentic_rag.evaluation import judge_answer

evidence = "\n".join(f"[{s.source_id}] {s.label}: {s.quote}" for s in advanced.sources)
verdict = judge_answer(settings, QUESTION, advanced.answer, evidence)
print(verdict.summary())
print("Critique:", verdict.scores.critique or "(none)")

100.0/100 PASS (ground 100, rel 100, compl 100, ctx 100)
Critique: (none)


## 5. The same question through the multi-agent graph
The Coordinator plans (high reasoning effort), the HR Policy agent runs its RAG tools, the Coordinator synthesises, and the judge gates the answer.

In [6]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

CRM MCP server at http://localhost:8765/mcp: customer=['get_customer_profile', 'search_customer_profiles'], ticket_read=['get_tickets_by_customer', 'search_tickets'], ticket_write=['register_ticket', 'resolve_ticket']
- `hr_policy` (available): HR policies (leave, travel and expenses, medical insurance, conduct, POSH, WFH, IT security, privacy, attendance, appraisal, whistleblowing, anti-bribery, notice period). Current versions by default; flags superseded values.
- `product_manuals` (available): TV and phone user manuals: specifications, setup, error codes, troubleshooting steps, warranty and safety. Pins the exact model (near-duplicate manuals).
- `customer_support` (available): Find customers by ID, name, email or phone and report their profile: active status, credit limit, contact details, remarks such as products owned. Read-only.
- `ticket_management` (available): Support ticket history and search (by customer, status, priority, product model, text). Proposes new tickets or reso

In [7]:
result = await service.ask(QUESTION, user_id="emp-priya", on_event=print)
result

[   0.0s] start_turn      done
[   0.0s] load_memory     0 long-term facts recalled
[   4.6s] plan            Initial plan for this question. -> s1=hr_policy
[  26.3s] run_step        s1 hr_policy done in 21.7s (confidence 0.80, 3 evidence, 0 proposed actions)
[  30.7s] reflect         synthesize: The HR policy report directly states the annual casual leave entitlement (8 days) and that unused casual leave lapses at
[  36.6s] synthesize      done
[  39.0s] judge           attempt 1: 99.3/100 PASS (ground 98, rel 100, compl 100, ctx 100)
[  39.0s] finalize        done
[  40.5s] save_memory     done


**status** `completed` | **thread** `thread-05131e32` | **elapsed** 40.5s  
**judge** 99.3/100 | **passed** True | **attempts** 1 | **specialist runs** 1

```text
plan round 1 (Initial plan for this question.): s1:hr_policy
timeline:
    s1 hr_policy             0.0s ->   21.7s |########################################|
judge attempt 1: 99.3/100 PASS (ground 98, rel 100, compl 100, ctx 100)
```

---

You’re entitled to **8 days of Casual Leave (CL) per calendar year**, and **you cannot carry them forward**.

- **Entitlement:** 8 days of CL per year, **credited on 1 January** each calendar year under the current leave policy HR-POL-001 v3.0. [E1][E2]  
- **Carry forward:** Any **unused CL lapses on 31 December** and **cannot be carried forward** to the next year. [E1]

If your employment type is anything other than confirmed full-time (e.g., part-time, contractor), check with HR, as the detailed rules for those categories weren’t in the retrieved excerpt.

In [8]:
# Close the memory databases
await service.aclose()

## What to observe
- Hybrid search may rank the **superseded** v2.0 table highly - correct retrieval, wrong policy version.
- Advanced RAG adds `status = current` unless the question asks about an old version; the explicit `v2.0` question is
  answered from v2.0 and says so.
- The graph answer cites evidence `[E#]`; the judge score is shown with the result.